# iPhone RGB-D reconstruction on Colab A100

Reusable notebook template. Upload a separately prepared `iphone-colab-input.zip` containing this package plus `iphone-input/capture.npz`, `board_calibration.json`, and `input_manifest.json`. Private recordings and model weights are not included in the repository. Use Colab Secrets for `HF_TOKEN`.

The model reconstructs one selected reference frame. This notebook explicitly uses ambiguity skipping and geometry fallback; failed meshes remain rejected and their reasons are retained. The real-capture baseline generated 14 meshes and accepted none. See `docs/STATUS.md`.

Select A100. Two workers require an 80 GB A100; use one on a 40 GB A100. Download results before the temporary runtime expires. No hand-motion reconstruction or robot manipulation is run.


In [ ]:
import os, sys, json, subprocess, signal, time, zipfile
from pathlib import Path
import torch
from google.colab import userdata, files
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Select A100; do not substitute L4/T4"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["MUJOCO_GL"] = "egl"
print({"gpu":torch.cuda.get_device_name(0),"credential_available":bool(os.environ["HF_TOKEN"])})


In [ ]:
uploaded = files.upload()
assert "iphone-colab-input.zip" in uploaded
PACKAGE = Path("/content/iphone-package-" + time.strftime("%Y%m%d-%H%M%S"))
PACKAGE.mkdir()
with zipfile.ZipFile("iphone-colab-input.zip") as archive:
    for member in archive.infolist():
        if not (PACKAGE/member.filename).resolve().is_relative_to(PACKAGE.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(PACKAGE)
del uploaded
subprocess.run([sys.executable,"-m","pip","install","uv","huggingface_hub","-e",str(PACKAGE)],check=True)
ROOT = Path("/content/roboskills")
COMMIT = "eca4691d2dfbb808dd58f3dd279fed154de6828a"
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/openrobokit/roboskills.git", str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
RUN = Path("/content/iphone-run-" + time.strftime("%Y%m%d-%H%M%S"))
RUN.mkdir()
print((PACKAGE/"iphone-input/input_manifest.json").read_text())


In [ ]:
def execute(name, argv, timeout=1800):
    log = RUN/(name+".log")
    print(name,"starting",flush=True)
    with log.open("w") as f:
        process = subprocess.Popen(list(map(str,argv)),stdout=f,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            rc=process.wait(timeout=timeout)
        except BaseException:
            os.killpg(process.pid,signal.SIGKILL)
            process.wait()
            raise
    if rc:
        print(log.read_text(errors="replace")[-5000:].replace(os.environ["HF_TOKEN"],"[REDACTED]"))
        raise RuntimeError(name+" failed; inspect saved log")
    print(name,"passed",flush=True)
native=PACKAGE/"scripts/colab_sam3d.py"
execute("setup",[sys.executable,native,"setup"])
execute("weights",[sys.executable,native,"weights"])


In [ ]:
execute("iphone-learned",[sys.executable,"-m","lidar_video.cli","chess-reconstruct-gpu",
    PACKAGE/"iphone-input/capture.npz","--calibration",PACKAGE/"iphone-input/board_calibration.json",
    "--reference-frame",0,"--ambiguous-policy","skip","--workers",2,"--native-script",native,"--mesh-policy","fallback","--out",RUN/"learned"],timeout=14400)


In [ ]:
from IPython.display import Video, display
result=RUN/"learned/result"
print((result/"verification.json").read_text())
display(Video(str(result/"simulation/rollout/mujoco_ext1.mp4"),embed=True))


In [ ]:
import shutil
for path in RUN.rglob("*.log"):
    path.write_text(path.read_text(errors="replace").replace(os.environ["HF_TOKEN"],"[REDACTED]"))
files.download(shutil.make_archive(str(RUN),"zip",RUN))
